# EDA Walkthrough
This notebook uses the same `eda` package that powers the Streamlit app, so every result here can be reproduced in the UI.

In [1]:
import sys; sys.path.append('..')
import pandas as pd, numpy as np
import eda
from eda import plots
df = eda.load_dataset('../data/sample_retail_orders.csv')
df.head()

,order_id,order_date,region,category,unit_price,quantity,discount_pct,customer_age,revenue,customer_rating
0,1,2024-02-18,North,Beauty,50.33,5,0,44.0,251.65,3.5
1,2,2025-02-21,South,Sports,172.86,5,0,NaN,864.30,3.7
2,3,2024-12-19,North,Beauty,70.73,3,0,50.0,212.19,1.9
3,4,2024-08-24,North,Home,20.87,1,0,29.0,20.87,2.8
4,5,2024-08-21,East,Home,77.34,5,0,53.0,386.70,4.7


## 1. Overview & health score

In [2]:
print(eda.dataset_overview(df))
eda.health_score(df)

{'rows': 2025, 'columns': 10, 'memory_mb': np.float64(0.177), 'numeric_cols': 7, 'categorical_cols': 2, 'datetime_cols': 1, 'missing_cells': 296, 'duplicate_rows': 25}


{'score': np.float64(94.6),
 'grade': 'A',
 'components': {'missing_pct': np.float64(1.46),
  'duplicate_pct': np.float64(1.23),
  'constant_col_pct': 0.0}}

## 2. Missing values

In [3]:
miss = eda.missing_report(df)
miss

,column,missing_count,missing_pct,severity
0,customer_age,161,7.95,Moderate
1,customer_rating,93,4.59,Low
2,region,42,2.07,Low


In [4]:
plots.missing_bar(miss)

## 3. Duplicates

In [5]:
eda.duplicate_report(df)['duplicate_rows']

25

## 4. Outlier profiling (IQR)

In [6]:
out = eda.outlier_summary(df)
out

,column,outliers,outlier_pct,lower_bound,upper_bound,min,max
0,revenue,69,3.41,-336.08,1022.48,7.84,7535.52
1,unit_price,50,2.47,-28.68,220.60,8.84,354.34
2,customer_age,20,1.07,7.50,67.50,4.00,140.00
3,quantity,0,0.00,-4.00,12.00,1.00,7.00
4,order_id,0,0.00,-990.00,2994.00,1.00,2000.00
5,discount_pct,0,0.00,-15.00,25.00,0.00,20.00
6,customer_rating,0,0.00,0.70,5.50,1.00,5.00


In [7]:
plots.box(df, 'revenue', 'category')

**Finding:** `revenue` has a heavy right tail and `customer_age` contains impossible values (e.g. 140). These should be capped or investigated before modelling.

## 5. Correlations

In [8]:
corr = eda.correlation_matrix(df)
plots.correlation_heatmap(corr)

In [9]:
eda.top_correlations(corr, 5)

,feature_1,feature_2,correlation
0,quantity,revenue,0.456807
1,unit_price,revenue,0.416903
2,customer_age,customer_rating,0.139846
3,discount_pct,customer_rating,-0.119314
4,discount_pct,revenue,-0.057899


## 6. Distributions & relationships

In [10]:
plots.histogram(df, 'unit_price', color='category')

In [11]:
plots.scatter(df, 'unit_price', 'revenue', color='region', trendline=True)

In [12]:
plots.time_series(df, 'order_date', 'revenue')

## 7. Cleaning decisions
1. Drop exact duplicate orders
2. Replace impossible ages (>100) with NaN then median-impute
3. Winsorise revenue at IQR fences

In [13]:
clean = df.drop_duplicates().copy()
clean.loc[clean.customer_age > 100, 'customer_age'] = np.nan
clean['customer_age'] = clean.customer_age.fillna(clean.customer_age.median())
print(eda.health_score(df)['score'], '->', eda.health_score(clean)['score'])

94.6 -> 98.7
